# Setup

In [37]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [38]:
import os
import shutil
import warnings
from functools import partial
from pprint import pprint
from typing import Callable

from tqdm.auto import tqdm

tqdm.pandas()
os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))
os.environ["TOKENIZERS_PARALLELISM"] = "false"

warnings.filterwarnings("ignore", category=FutureWarning)

In [39]:
import numpy as np
import pandas as pd
from datasets import DatasetDict
from semhash import SemHash
from sentence_transformers import SentenceTransformer, util

np.random.seed(0)

In [40]:
from experiments.constants import INFINITY_INSTRUCT__CONFIG
from experiments.languages import LanguageClassifier
from experiments.utils import get_final_stats
from experiments.utils import load_dataset as src_load_dataset
from experiments.utils import save_json, save_parquet, split_into_sentences

Embedding model is later used to search for labels in dataset that are similar to pre-defined terms. Model has been chosen based on official docs, claimed to be fast and to perform well on small terms - our search is one-word mostly.

In [41]:
language_classifier: LanguageClassifier = LanguageClassifier()
load_dataset: Callable[[str], DatasetDict] = partial(src_load_dataset, config=INFINITY_INSTRUCT__CONFIG)
embedding_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")

In [42]:
FRENCH_LANGUAGE_CODE: str = "fr"
ENGISH_LANGUAGE_CODE: str = "en"
SPANISH_LANGUAGE_CODE: str = "es"

# Text dataset preparation and EDA

In [43]:
ds: DatasetDict = load_dataset(INFINITY_INSTRUCT__CONFIG.configs["660k"])
pprint(ds)

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'conversations', 'label', 'langdetect', 'source'],
        num_rows: 659808
    })
})


In [44]:
pprint(ds["train"].info)

DatasetInfo(description='',
            citation='',
            homepage='',
            license='',
            features={'conversations': [{'from': Value(dtype='string', id=None),
                                         'value': Value(dtype='string',
                                                        id=None)}],
                      'id': Value(dtype='string', id=None),
                      'label': {'ability_en': Sequence(feature=Value(dtype='string',
                                                                     id=None),
                                                       length=-1,
                                                       id=None),
                                'ability_zh': Sequence(feature=Value(dtype='string',
                                                                     id=None),
                                                       length=-1,
                                                       id=None),
                           

In [45]:
pprint(ds["train"][0])

{'conversations': [{'from': 'human',
                    'value': 'In a certain country populated predominantly by '
                             'wizards, there is an impending demonstration '
                             'that needs careful planning to ensure it meets '
                             'specific requirements.\n'
                             '\n'
                             'There is a population of n individuals within '
                             'the city, of which x are wizards committed to '
                             'attending the demonstration. The remaining (n - '
                             'x) individuals, non-wizards, have no intention '
                             'to participate. The city administration will '
                             'acknowledge and act upon the demonstration only '
                             'if it engages at least y percent of the total '
                             'city population. To meet this criterion, wizards '
      

Conversations are renamed to prompt for compatibility with VoiceBench dataset.

In [46]:
df = pd.DataFrame(
    [
        [
            e["conversations"],
            e["label"].get("ability_en") if isinstance(e["label"], dict) else None,
            e["langdetect"],
            e["source"],
            e["id"],
        ]
        for e in ds["train"]
    ],
    columns=["prompt", "labels", "language", "source", "id"],
)

In [47]:
LANGUAGE_MINIMAL_POPULATION: int = 500
df["language"].value_counts()[lambda x: x >= LANGUAGE_MINIMAL_POPULATION].reset_index()

,language,count
0,en,641610
1,zh-cn,7972
2,es,1453
3,fr,1415
4,ko,1308
5,pt,999
6,it,797
7,de,544
8,ru,500


In [48]:
df = df[df["language"].isin(INFINITY_INSTRUCT__CONFIG.languages)].reset_index(drop=True)
df["speakers"] = df["prompt"].apply(lambda x: {el["from"] for el in x})

print(f"Percentage of unique IDs: {(len(df.drop_duplicates(subset=['id'])) / len(df)) * 100:.2f}%")
print(f"Max speakers in a conversation: {df['speakers'].map(len).max()}")

Percentage of unique IDs: 100.00%
Max speakers in a conversation: 2


In [49]:
df["prompt"] = df["prompt"].apply(
    lambda x: [el["value"] for el in x if el["from"] == "human"]  # pylint: disable=magic-value-comparison
)

df["n_turns"] = df["prompt"].str.len()
print(f"Average number of turns: {df['n_turns'].mean():.2f}")
print(f"Max number of turns: {df['n_turns'].max():.2f}")

Average number of turns: 1.12
Max number of turns: 94.00


In [50]:
MAXIMUM_NUMBER_OF_TURNS: int = 10
original_num = len(df)
df = df[df["n_turns"] <= MAXIMUM_NUMBER_OF_TURNS].reset_index(drop=True)
print(f"Remaining samples after filtering long prompts: {len(df)} (removed {original_num - len(df)})")

Remaining samples after filtering long prompts: 643776 (removed 702)


In [51]:
unique_keys = df["labels"].explode().unique().astype(str)
unique_keys = unique_keys[unique_keys != ""]
unique_keys.sort()
print(unique_keys)

['(' ')' '1' ... 'zoological knowledge' '背景设定' '音声理解']


As we operate on small models, we exclude math-based problems. Moreover, to minimize number of out-of-box tokens, we remove programming related entries.

In [52]:
ABILITIES_TO_EXCLUDE: set[str] = {
    "formal logic",
    "logical reasoning",
    "mathematics",
    "programming",
}
THRESHOLD: float = 0.7

term_embeddings = embedding_model.encode(unique_keys, convert_to_tensor=True)
exclude_embeddings = embedding_model.encode(list(ABILITIES_TO_EXCLUDE), convert_to_tensor=True)
similarity_matrix = util.cos_sim(term_embeddings, exclude_embeddings)

keys_to_exclude = {
    str(key)
    for i, key in enumerate(unique_keys)
    if any(similarity_matrix[i][j] > THRESHOLD for j in range(len(ABILITIES_TO_EXCLUDE)))
}
keys_to_include = set(unique_keys) - keys_to_exclude
original_num = len(df)
df = df[df["labels"].apply(lambda x: all(el in keys_to_include for el in x))].reset_index(drop=True)
pprint(sorted(keys_to_exclude))
print(
    f"Remaining samples after filtering abilities: {len(df)} "
    f"(removed {original_num - len(df)}, {((original_num - len(df)) / original_num) * 100:.2f}%)"
)

['advanced mathematics',
 'arduino programming',
 'basic mathematics',
 'conditional reasoning',
 'formal language',
 'formal logic',
 'formal logic reasoning',
 'formal reasoning',
 'java programming',
 'latex programming',
 'logic debate',
 'logic design',
 'logic explanation',
 'logic processing',
 'logical construction',
 'logical expression',
 'logical operation',
 'logical reasoning',
 'logical reasoning ability',
 'logical thinking',
 'mathematical analysis',
 'mathematical concept explanation',
 'mathematical concepts',
 'mathematical knowledge',
 'mathematical skills',
 'mathematical theory',
 'multiple reasoning',
 'program design',
 'program writing',
 'programming ability',
 'programming explanation',
 'programming knowledge',
 'programming technology',
 'propositional logic reasoning',
 'reasoning',
 'solidity programming',
 'suggestion for reasoning',
 'symbolic logic reasoning',
 'system programming']
Remaining samples after filtering abilities: 423657 (removed 220119, 3

Following cell performs deduplication based on joined prompt messages using semhash.

In [53]:
df["prompts_joined"] = df["prompt"].str.join(" ")

# construct index
semhash = SemHash.from_records(records=df["prompts_joined"].tolist())

# perform deduplication and filtering of outliers
dedup_texts = set(semhash.self_deduplicate().selected)
filtered_texts = {e["text"] for e in semhash.self_filter_outliers().selected}
representative_texts = dedup_texts.intersection(filtered_texts)

print(
    f"Number of unique texts after deduplication: {len(filtered_texts)} "
    f"(removed {len(df) - len(filtered_texts)}, {(len(df) - len(filtered_texts)) / len(df) * 100:.2f}%)"
)
df = df[df["prompts_joined"].isin(representative_texts)].reset_index(drop=True)

Number of unique texts after deduplication: 379488 (removed 44169, 10.43%)


Having removed 10% of dataset length, we're now only around 750 rows for non-english subsets.

In [54]:
df["language"].value_counts()

language
en    324855
fr       759
es       757
Name: count, dtype: int64

Next step is to split prompt's messages into separate sentences using nltk, in same way as we have in VoiceBench.

In [55]:
df.drop(columns=["id", "speakers", "source"], inplace=True)
df["prompt"] = df["prompt"].apply(lambda x: [split_into_sentences(e.strip()) for e in x if e.strip() != ""])
df.head()

,prompt,labels,language,n_turns,prompts_joined
0,[[Compose a 1500-word analytical essay formatt...,"[literature search skills, writing skills, inf...",en,1,Compose a 1500-word analytical essay formatted...
1,[[Analyze the two tables presented below conce...,"[problem solved, data analysis]",en,1,Analyze the two tables presented below concern...
2,[[The period from the appearance of the first ...,"[unit conversion, general knowledge about scie...",en,1,The period from the appearance of the first s...
3,[[Can you explain how to properly use a circul...,"[problem solved, skill advice, knowledge quest...",en,7,Can you explain how to properly use a circular...
4,[[Create a vividly detailed and provocative MC...,[content filtering],en,1,Create a vividly detailed and provocative MCU ...


In [56]:
df[df["language"] == FRENCH_LANGUAGE_CODE].head(10).reset_index(drop=True)

,prompt,labels,language,n_turns,prompts_joined
0,[[Nommez NAME_1 les meilleurs joueurs de footb...,"[multicultural understanding, data analysis, i...",fr,1,Nommez NAME_1 les meilleurs joueurs de footbal...
1,[[Read the following dialogue in French and su...,"[problem solved, translation, analysis and rea...",fr,1,Read the following dialogue in French and sum...
2,[[pouvez-vous nommer la capitale du pays NAME_...,"[multicultural understanding, search knowledge...",fr,1,pouvez-vous nommer la capitale du pays NAME_1 ?
3,"[[Comment traduire ""We Have No Moat"" en frança...","[translation, oral communication]",fr,1,"Comment traduire ""We Have No Moat"" en français..."
4,[[Question : Rédigez en français un article in...,"[text generation, search knowledge, text editi...",fr,1,Question : Rédigez en français un article info...
5,"[[### Human: Your task is to classify texts., ...","[text classification, categorization and tags]",fr,1,### Human: Your task is to classify texts.\n\n...
6,[[Traduzca el siguiente documento al francés: ...,"[project management, legal knowledge, translat...",fr,1,"Traduzca el siguiente documento al francés: ""L..."
7,[[Vous êtes un banquier NAME_1 à la banque cen...,"[text generation, translation, writing skills,...",fr,1,Vous êtes un banquier NAME_1 à la banque centr...
8,[[Pourriez-vous rédiger en français une descri...,"[multicultural understanding, text generation,...",fr,1,Pourriez-vous rédiger en français une descript...
9,[[Réécris moi ça en ajoutant des trucs pour la...,[text-to-speech],fr,2,Réécris moi ça en ajoutant des trucs pour la p...


There are some problems with non-english languages labels (line 5), we'll filter them out.

In [57]:
df["detected_lang"] = None
non_en_mask = df["language"] != ENGISH_LANGUAGE_CODE
df.loc[non_en_mask, "detected_lang"] = df.loc[non_en_mask, "prompts_joined"].progress_apply(
    lambda x: language_classifier.classify_language(x[:500])
)
df["detected_lang"].value_counts()

  0%|          | 0/1516 [00:00<?, ?it/s]

Device set to use mps:0


detected_lang
es    725
fr    601
en    129
th     22
pt     15
zh     12
it      5
hi      3
ur      2
sw      1
ru      1
Name: count, dtype: int64

Around 10% of non-english entries were incorrectly classified, for safety we'll remove them.

In [58]:
incorrect_mask = (df["language"] != df["detected_lang"]) & df["detected_lang"].notna()
print(
    f"Number of incorrect language detections: {incorrect_mask.sum()} "
    f"({(incorrect_mask.sum() / non_en_mask.sum()) * 100:.2f}%)"
)

Number of incorrect language detections: 194 (12.80%)


In [59]:
df.drop(incorrect_mask[incorrect_mask].index, inplace=True)
df.drop(columns=["detected_lang"], inplace=True)

## Multilingual

For that subset we pick only 1-turn conversations, 500 messages from each of 3 languages: fr, en and es.

### Dataset preparation

In [60]:
multilingual__df = (
    df[df["n_turns"] == 1]
    .groupby("language", as_index=False)
    .apply(lambda g: g.sample(n=min(len(g), 500), random_state=0))
    .reset_index(drop=True)
)
multilingual__df["prompt"] = multilingual__df["prompt"].apply(lambda x: x[0])
multilingual__df_to_save = multilingual__df.drop(columns=["prompts_joined", "n_turns"])
save_parquet(multilingual__df_to_save, INFINITY_INSTRUCT__CONFIG.data_dir / "text__multi_lingual.parquet")

In [61]:
multilingual__df["language"].value_counts()

language
en    500
es    500
fr    500
Name: count, dtype: int64

### Stats for processed data

In [62]:
get_final_stats(multilingual__df, prompt_key="prompts_joined")

,num_rows,total_characters,avg_num_characters,total_sentences,avg_num_sentences,unique_entries,pct_unique_entries
0,1500,465284,310.189333,3633,2.422,1500,100.0


### Processed samples

In [63]:
sample__df = multilingual__df_to_save.groupby("language").sample(n=1, random_state=0).reset_index(drop=True)
save_json(sample__df, INFINITY_INSTRUCT__CONFIG.data_dir / "text__multilingual__samples.json")
sample__df.head(3)

,prompt,labels,language
0,"[""Using Python, write a code that generates a ...","[function writing, python programming, file op...",en
1,[¿Cómo puedo montar un clúster de kubernetes e...,"[problem solved, code execution, technical gui...",es
2,"[En tant que parent, quelles questions détaill...","[medical consultation, problem solved, informa...",fr


## Multi turn

For that dataset we choose only conversations with 6 turns in english.

### Dataset preparation

In [64]:
df["n_turns"].value_counts()

n_turns
1     313066
3       3353
4       3056
2       2574
5       1453
6       1249
7       1159
8        116
9         88
10        63
Name: count, dtype: int64

In [65]:
NUMBER_OF_TURNS_TO_INCLUDE: int = 6
multi_turn__df = (
    df[(df["language"] == ENGISH_LANGUAGE_CODE) & (NUMBER_OF_TURNS_TO_INCLUDE == df["n_turns"])]
    .sample(500, random_state=0)
    .reset_index(drop=True)
)
multi_turn__df_to_save = multi_turn__df.drop(columns=["prompts_joined", "n_turns", "language"])
save_parquet(multi_turn__df_to_save, INFINITY_INSTRUCT__CONFIG.data_dir / "text__multi_turn.parquet")
len(multi_turn__df)

500

### Stats for processed data

In [66]:
get_final_stats(multi_turn__df, prompt_key="prompts_joined")

,num_rows,total_characters,avg_num_characters,total_sentences,avg_num_sentences,unique_entries,pct_unique_entries
0,500,507064,1014.128,5989,11.978,500,100.0


### Processed samples

In [67]:
sample__df = multi_turn__df_to_save.sample(n=1, random_state=0).reset_index(drop=True)
save_json(sample__df, INFINITY_INSTRUCT__CONFIG.data_dir / "text__multi_turn__samples.json")
sample__df.head(3)

,prompt,labels
0,[[Discuss the significance of Achilles' decisi...,"[historical knowledge, character analysis, ana..."


# Cleanup

In [ ]:
shutil.rmtree(INFINITY_INSTRUCT__CONFIG.cache_dir)